### Build a Simple LLM Application with LCEL
In this quickstart we'll show you how to build a simple LLM application with LangChain. This application will translate text from English into another language. This is a relatively simple LLM application - it's just a single LLM call plus some prompting. Still, this is a great way to get started with LangChain - a lot of features can be built with just some prompting and an LLM call!

After seeing this video, you'll have a high level overview of:

- Using language models

- Using PromptTemplates and OutputParsers

- Using LangChain Expression Language (LCEL) to chain components together

- Debugging and tracing your application using LangSmith

- Deploying your application with LangServe

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

import openai
openai.api_key = os.getenv("OPENAI_API_KEY")

groq_api_key = os.getenv("GROQ_API_KEY")

In [5]:
from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq

model=ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", groq_api_key=groq_api_key)
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B7C546C320>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B7C546C7D0>, model_name='meta-llama/llama-4-scout-17b-16e-instruct', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [9]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [ 
    SystemMessage(content="Translate the following from English to French."),
    HumanMessage(content="Hello, how are you?")
]

response = model.invoke(messages)

In [12]:
from langchain_core.output_parsers import StrOutputParser
output_parser = StrOutputParser()

output_parser.invoke(response)

'Bonjour, comment allez-vous?'

In [13]:
### Using LCEL we can chain the components.

chain = model | output_parser
chain.invoke(messages)

'Bonjour, comment allez-vous ?'

In [48]:
### Prompt Templates are also supported, instead parsing messages.
from langchain_core.prompts import ChatPromptTemplate

generic_template="Translate the following into {language}. Reply only the translation"

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", generic_template),
        ("user", "{text}")
    ]
)

In [49]:
result = prompt.invoke({"language": "French", "text": "Hello, how are you?"})

In [50]:
result.to_messages()

[SystemMessage(content='Translate the following into French. Reply only the translation', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={})]

In [56]:
chain=prompt | model | output_parser
chain.invoke({"language": "Brazilian Portuguese", "text": "Hello, how are you?"})

'Olá, como você está?'